In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv

from ulac.common import mesh_utils
from ulac.mapping import basis, jacobian

In [ ]:
patient_id = "01"
input_mesh_file = Path(f"../data/patient_{patient_id}/mesh_with_fibers_tags.vtk")
anatomical_tags = {"MV": 0, "LAA": 1, "LIPV": 2, "LSPV": 3, "RIPV": 4, "RSPV": 5}

mesh = pv.read(input_mesh_file)
mesh = mesh_utils.convert_unstructured_to_polydata_mesh(mesh)
simplices = np.array(mesh.faces.reshape(-1, 4)[:, 1:4])

In [ ]:
mvc_map = basis.construct_smooth_parameterization(
    mesh, exterior_boundary_tag=anatomical_tags["MV"], exterior_boundary_radius=1000.0
)
z_coordinates = np.zeros(len(mvc_map))
uacs_points = np.column_stack((mvc_map, z_coordinates))
plotter = pv.Plotter()
uac_mesh = pv.PolyData.from_regular_faces(uacs_points, simplices)
plotter.add_mesh(uac_mesh, style="wireframe", color="grey")
plotter.show()

In [ ]:
cellwise_jacobians = jacobian.compute_cellwise_jacobians(
    vertices_3d=mesh.points, vertices_2d=mvc_map, simplices=simplices
)